In [ ]:
%matplotlib inline
import sys, os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, '/Users/yotameviatar/vs_code')

from Airflow import airflow_config as cfg
from Airflow import airflow_processor as processor

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.alpha': 0.35, 'grid.linestyle': '--'})
print('Setup complete')

In [ ]:
# ── Configuration overrides ───────────────────────────────────────────────────
SUBJECT_FILTER             = []
ANAL_TMIN                  = -5.0
ANAL_TMAX                  =  10.0
BASELINE_TMIN              = -5.0
BASELINE_TMAX              =  0.0
RESPONSE_TMIN              =  0.0   # startle gasp begins at t=0
RESPONSE_TMAX              =  3.0   # gasp resolves within 3 s
USE_SUBJECTIVE_TRIAL_TYPE  = False   # must match hr_config.py and EMG (objective: image_type column)
AIRFLOW_Z_SCORE_THRESHOLD  = 3.0    # baseline noise gate; None = disabled
AIRFLOW_SCORE_MAX          = 5.0    # score in session-breath-amplitude units; >5 = artefact
PERFORM_SCORING            = True

# RSP_CLEAN_METHOD is a Layer 1 (cache) parameter — changing it here has no effect
# unless the cache is rebuilt. Shown for documentation only.
RSP_CLEAN_METHOD = "khodadad2018"   # Butterworth bandpass 0.05–3 Hz

# ── Rejection gates ───────────────────────────────────────────────────────────
AIRFLOW_AMPLITUDE_Z_THRESHOLD = 4.0   # robust z-score (median/MAD) on max|epoch|
AIRFLOW_MIN_STD_RATIO         = 0.1   # flat signal: baseline std < ratio × session median
RSP_RATE_ARTIFACT_THRESHOLD   = 40    # bpm; NK2 RSP_Rate > this → movement artefact

# ── Display options ───────────────────────────────────────────────────────────
NORMALIZE_TIMECOURSE   = True   # z-score each session trace for uniform y-scale across subjects
IQR_OUTLIER_MULTIPLIER = 2.0    # paired IQR fence multiplier for boxplot outlier removal

In [ ]:
# ── Load cache & run Layer 2 analysis ────────────────────────────────────────
# Layer 1 (cache) = raw signal at 25 Hz + CSV metadata. Never re-read unless
# AIRFLOW_CHANNEL or CACHE_SFREQ changes (requires FORCE_RELOAD in airflow_config.py).
#
# Layer 2 (here) = NK2 filter → epoch cutting → baseline → rejection → scoring.
# Change any parameter in cell-1 and re-run this cell. No MFF access needed.

CACHE_FILE = os.path.join(cfg.OUTPUT_DIR, '_cache', 'airflow_cache.pkl')
with open(CACHE_FILE, 'rb') as f:
    _cache = pickle.load(f)

_sessions = _cache.get('sessions', {})
if SUBJECT_FILTER:
    _sessions = {k: v for k, v in _sessions.items() if k in SUBJECT_FILTER}

# Push overrides to cfg before Layer 2 runs
cfg.ANAL_TMIN                     = ANAL_TMIN
cfg.ANAL_TMAX                     = ANAL_TMAX
cfg.BASELINE_TMIN                 = BASELINE_TMIN
cfg.BASELINE_TMAX                 = BASELINE_TMAX
cfg.RESPONSE_TMIN                 = RESPONSE_TMIN
cfg.RESPONSE_TMAX                 = RESPONSE_TMAX
cfg.USE_SUBJECTIVE_TRIAL_TYPE     = USE_SUBJECTIVE_TRIAL_TYPE
cfg.AIRFLOW_Z_SCORE_THRESHOLD     = AIRFLOW_Z_SCORE_THRESHOLD
cfg.AIRFLOW_SCORE_MAX             = AIRFLOW_SCORE_MAX
cfg.PERFORM_SCORING               = PERFORM_SCORING
cfg.AIRFLOW_AMPLITUDE_Z_THRESHOLD = AIRFLOW_AMPLITUDE_Z_THRESHOLD
cfg.AIRFLOW_MIN_STD_RATIO         = AIRFLOW_MIN_STD_RATIO
cfg.RSP_RATE_ARTIFACT_THRESHOLD   = RSP_RATE_ARTIFACT_THRESHOLD
cfg.RSP_CLEAN_METHOD              = RSP_CLEAN_METHOD

trials_data, _traces = processor.apply_analysis_params(_sessions, cfg)

print(f'Loaded {len(trials_data)} subjects')
for subj, sessions in sorted(trials_data.items()):
    for sk, trials in sessions.items():
        n_rej = sum(t['rejected'] for t in trials)
        print(f'  {subj} {sk}: {len(trials)} trials, {n_rej} rejected')
        for i, t in enumerate(trials):
            reason = t.get('rejection_reason', '')
            if t['rejected'] and reason:
                print(f'      trial {i+1:02d}: {reason}')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def _sig(p):
    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

def _bracket(ax, x1, x2, y_top, p, label):
    h = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.015
    ax.plot([x1, x1, x2, x2], [y_top, y_top+h, y_top+h, y_top], lw=1.2, color='k')
    ax.text((x1+x2)/2, y_top+h*1.5, f'{_sig(p)}  {label}  p={p:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

def _iqr_filter_paired(scores_a, scores_b, subjects, multiplier=2.0):
    """Remove paired outliers by IQR fence on the (a − b) difference."""
    if len(scores_a) < 4:
        return list(scores_a), list(scores_b), list(subjects), 0
    diffs = np.array(scores_a) - np.array(scores_b)
    q1, q3 = np.percentile(diffs, 25), np.percentile(diffs, 75)
    iqr = q3 - q1
    ok = (diffs >= q1 - multiplier * iqr) & (diffs <= q3 + multiplier * iqr)
    n_removed = int(np.sum(~ok))
    return ([s for s, v in zip(scores_a, ok) if v],
            [s for s, v in zip(scores_b, ok) if v],
            [s for s, v in zip(subjects, ok) if v],
            n_removed)

EVE_COLOR  = '#E67E22'
MOR_COLOR  = '#8E44AD'
UP_COLOR   = '#27AE60'
DOWN_COLOR = '#E74C3C'
# Score is now normalised excursion: response peak deviation / session breath amplitude.
# 1.0 = response as large as a typical breath; 2.0 = twice as large.
YLABEL = 'Normalised excursion (breath amplitudes)'

In [ ]:
# ── Session timecourse — full trace with startle markers ─────────────────────
# _traces is produced by apply_analysis_params (the NK2-filtered session signal).
# sharey=True locks Eve and Mor to the same y-axis so signal quality is comparable.
for subj, sessions in sorted(trials_data.items()):
    fig, axes = plt.subplots(2, 1, figsize=(18, 6), sharex=False, sharey=True)
    fig.suptitle(f'{subj} — Airflow Session Timecourse ({RSP_CLEAN_METHOD})',
                 fontsize=12, fontweight='bold')

    for ax, sess_key in zip(axes, ['eve', 'mor']):
        trace  = _traces.get(subj, {}).get(sess_key)
        trials = sessions.get(sess_key, [])
        if trace is None or not trials:
            ax.set_title(f'{sess_key.upper()} — no data'); ax.axis('off'); continue

        signal        = trace['signal'].copy().astype(float)
        sfreq         = trace['sfreq']
        times         = np.arange(len(signal)) / sfreq
        startle_times = trace['startle_samps_sec']

        if NORMALIZE_TIMECOURSE:
            s_std = np.std(signal)
            if s_std > 0:
                signal = (signal - np.mean(signal)) / s_std
            ylabel = 'Airflow (z-score)'
        else:
            ylabel = 'Airflow (RSP_Clean)'

        ax.plot(times, signal, linewidth=0.8, color='steelblue', alpha=0.9)

        for i, t_sec in enumerate(startle_times):
            if i < len(trials):
                t_obj = trials[i]
                clr = cfg.REJ_COLOR if t_obj['rejected'] else (
                      cfg.NEG_COLOR if t_obj['label'] == 1 else cfg.NEU_COLOR)
            else:
                clr = 'gray'
            ax.axvline(t_sec, color=clr, linewidth=1.0, alpha=0.6)

        ax.set_title(cfg.SESSION_MAP[sess_key]['label'], fontsize=11, fontweight='bold')
        ax.set_xlabel('Time in session (s)')
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.legend(handles=[
            plt.Line2D([0],[0], color=cfg.NEG_COLOR, label='Negative'),
            plt.Line2D([0],[0], color=cfg.NEU_COLOR, label='Neutral'),
            plt.Line2D([0],[0], color=cfg.REJ_COLOR, label='Rejected'),
        ], fontsize=8, loc='upper right')
    plt.tight_layout(); plt.show()

In [ ]:
# ── Review trials — per-subject grid, shared y-axis ──────────────────────────
# One figure per subject per session. All trials on the same y-scale so bad
# signals are immediately visible. Rejected trials shown in grey, spikes clip.
#
# Y-scale is set from the 2nd–98th percentile of ACCEPTED trials only,
# so a few spike epochs don't compress all other signals into a flat line.

N_COLS = 6   # trials per row in the grid

for subj, sessions in sorted(trials_data.items()):
    # Collect ylim from accepted trials across BOTH sessions for this subject
    accepted_vals = []
    for sk in ['eve', 'mor']:
        for t in sessions.get(sk, []):
            if not t['rejected'] and t['epoch_clean_wide'] is not None:
                accepted_vals.append(t['epoch_clean_wide'])
    if accepted_vals:
        flat = np.concatenate(accepted_vals)
        ylo, yhi = np.percentile(flat, 2), np.percentile(flat, 98)
        margin = max((yhi - ylo) * 0.15, np.abs(yhi) * 0.05)
        ylim = (ylo - margin, yhi + margin)
    else:
        ylim = (-1, 1)

    for sess_key in ['eve', 'mor']:
        trials = sessions.get(sess_key, [])
        if not trials:
            continue

        n      = len(trials)
        n_rows = int(np.ceil(n / N_COLS))
        fig, axes = plt.subplots(n_rows, N_COLS,
                                  figsize=(N_COLS * 3.2, n_rows * 2.8),
                                  sharey=True)
        axes_flat = np.array(axes).flatten()

        n_rej = sum(t['rejected'] for t in trials)
        fig.suptitle(
            f'{subj} — {cfg.SESSION_MAP[sess_key]["label"]}  '
            f'({n} trials, {n_rej} rejected)',
            fontsize=12, fontweight='bold'
        )

        for i, t in enumerate(trials):
            ax  = axes_flat[i]
            tw  = t['times_wide']
            ep  = t['epoch_clean_wide']

            if t['rejected']:
                color = cfg.REJ_COLOR
                reason = t.get('rejection_reason', '')
                title  = f'T{i+1} REJ\n{reason[:18]}'
                lw     = 0.8
            elif t['label'] == 1:
                color = cfg.NEG_COLOR
                score_str = f'{t["score"]:.2f}' if t['score'] is not None and not np.isnan(t['score']) else '--'
                title = f'T{i+1} Neg  s={score_str}'
                lw    = 1.0
            else:
                color = cfg.NEU_COLOR
                score_str = f'{t["score"]:.2f}' if t['score'] is not None and not np.isnan(t['score']) else '--'
                title = f'T{i+1} Neu  s={score_str}'
                lw    = 1.0

            ax.plot(tw, ep, color=color, linewidth=lw, alpha=0.9)
            ax.axvline(0,  color='k',     linestyle='--', linewidth=0.7, alpha=0.6)
            ax.axvspan(cfg.BASELINE_TMIN, cfg.BASELINE_TMAX,
                       color='gold',  alpha=0.10, zorder=0)
            ax.axvspan(cfg.RESPONSE_TMIN, cfg.RESPONSE_TMAX,
                       color='green', alpha=0.07, zorder=0)
            ax.set_ylim(ylim)
            ax.set_title(title, fontsize=7, pad=2,
                         color='dimgray' if t['rejected'] else 'black')
            ax.tick_params(labelsize=5.5)
            ax.set_xlabel('s', fontsize=5)
            if i % N_COLS == 0:
                ax.set_ylabel('RSP', fontsize=6)

        for j in range(n, len(axes_flat)):
            axes_flat[j].axis('off')

        # Legend once in top-right unused panel or as a figure legend
        fig.legend(handles=[
            plt.Line2D([0],[0], color=cfg.NEG_COLOR, lw=1.5, label='Negative'),
            plt.Line2D([0],[0], color=cfg.NEU_COLOR, lw=1.5, label='Neutral'),
            plt.Line2D([0],[0], color=cfg.REJ_COLOR, lw=1.5, label='Rejected'),
        ], fontsize=8, loc='lower right', ncol=3, framealpha=0.9)

        plt.tight_layout(rect=[0, 0.03, 1, 0.96])
        plt.show()

In [ ]:
# ── Subject average timecourse (Eve | Mor) ────────────────────────────────────
for subj, sessions in sorted(trials_data.items()):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    fig.suptitle(f'{subj} — Average Airflow Timecourse', fontsize=12, fontweight='bold')

    for ax, sess_key in zip(axes, ['eve', 'mor']):
        trials = sessions.get(sess_key, [])
        neg_e, neu_e, times = [], [], None
        for t in trials:
            if t['rejected'] or t['epoch_anal'] is None: continue
            (neg_e if t['label'] == 1 else neu_e).append(t['epoch_anal'])
            if times is None: times = t['times_anal']
        if times is None:
            ax.set_title(f"{cfg.SESSION_MAP[sess_key]['label']} — no data"); continue
        for eps, clr, lbl in [(neg_e, cfg.NEG_COLOR, 'Negative'),
                               (neu_e, cfg.NEU_COLOR, 'Neutral')]:
            if not eps: continue
            arr = np.array(eps)
            avg = np.mean(arr, axis=0)
            sem = stats.sem(arr, axis=0, nan_policy='omit') if len(arr) > 1 else np.zeros_like(avg)
            ax.plot(times, avg, color=clr, linewidth=2.2, label=f'{lbl} (n={len(eps)})')
            ax.fill_between(times, avg-sem, avg+sem, color=clr, alpha=0.2)
        ax.axvline(0, color='k', linestyle='--', linewidth=1.2)
        ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
        ax.axvspan(RESPONSE_TMIN, RESPONSE_TMAX, color='green', alpha=0.07)
        ax.axvspan(BASELINE_TMIN, BASELINE_TMAX, color='gold',  alpha=0.07)
        ax.set_title(cfg.SESSION_MAP[sess_key]['label'], fontsize=11, fontweight='bold')
        ax.set_xlabel('Time (s)'); ax.set_ylabel('Airflow (RSP_Clean)'); ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Group average timecourse ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle('Group Average Airflow Timecourse', fontsize=13, fontweight='bold')

for ax, sess_key in zip(axes, ['eve', 'mor']):
    neg_e, neu_e, times = [], [], None
    n_neg_s = n_neu_s = 0
    for subj, sessions in trials_data.items():
        trials = sessions.get(sess_key, [])
        s_neg = [t['epoch_anal'] for t in trials
                 if not t['rejected'] and t['label'] == 1 and t['epoch_anal'] is not None]
        s_neu = [t['epoch_anal'] for t in trials
                 if not t['rejected'] and t['label'] == 2 and t['epoch_anal'] is not None]
        if s_neg: neg_e.append(np.mean(s_neg, axis=0)); n_neg_s += 1
        if s_neu: neu_e.append(np.mean(s_neu, axis=0)); n_neu_s += 1
        if times is None:
            times = next((t['times_anal'] for t in trials if t['times_anal'] is not None), None)
    if times is None:
        ax.set_title(f"{cfg.SESSION_MAP[sess_key]['label']} — no data"); continue
    for eps, clr, lbl, ns in [(neg_e, cfg.NEG_COLOR, 'Negative', n_neg_s),
                               (neu_e, cfg.NEU_COLOR, 'Neutral',  n_neu_s)]:
        if not eps: continue
        arr = np.array(eps)
        avg = np.mean(arr, axis=0)
        sem = stats.sem(arr, axis=0, nan_policy='omit') if len(arr) > 1 else np.zeros_like(avg)
        ax.plot(times, avg, color=clr, linewidth=2.2, label=f'{lbl} (n subj={ns})')
        ax.fill_between(times, avg-sem, avg+sem, color=clr, alpha=0.2)
    ax.axvline(0, color='k', linestyle='--', linewidth=1.2)
    ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    ax.axvspan(RESPONSE_TMIN, RESPONSE_TMAX, color='green', alpha=0.07)
    ax.axvspan(BASELINE_TMIN, BASELINE_TMAX, color='gold',  alpha=0.07)
    ax.set_title(cfg.SESSION_MAP[sess_key]['label'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Airflow (RSP_Clean)'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── Overall timecourse: Evening vs Morning ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
times = None
for sess_key, clr, lbl in [('eve', EVE_COLOR, 'Evening'),
                             ('mor', MOR_COLOR, 'Morning')]:
    subj_avgs = []
    for subj, sessions in trials_data.items():
        trials   = sessions.get(sess_key, [])
        accepted = [t for t in trials if not t['rejected'] and t['epoch_anal'] is not None]
        if not accepted: continue
        subj_avgs.append(np.mean([t['epoch_anal'] for t in accepted], axis=0))
        if times is None: times = accepted[0]['times_anal']
    if not subj_avgs or times is None: continue
    arr = np.array(subj_avgs)
    avg = np.mean(arr, axis=0)
    sem = stats.sem(arr, axis=0, nan_policy='omit') if len(arr) > 1 else np.zeros_like(avg)
    ax.plot(times, avg, color=clr, linewidth=2.5, label=f'{lbl} (n={len(subj_avgs)})')
    ax.fill_between(times, avg-sem, avg+sem, color=clr, alpha=0.2)
ax.axvline(0, color='k', linestyle='--', linewidth=1.2)
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.axvspan(RESPONSE_TMIN, RESPONSE_TMAX, color='green', alpha=0.07)
ax.axvspan(BASELINE_TMIN, BASELINE_TMAX, color='gold',  alpha=0.07)
ax.set_xlabel('Time (s)', fontsize=11); ax.set_ylabel('Airflow (RSP_Clean)', fontsize=11)
ax.set_title('Overall Average Airflow: Evening vs Morning (±SEM)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# ── Group score boxplot (4 conditions) ───────────────────────────────────────
cond_keys = [('eve', 1, 'Eve-Neg', cfg.NEG_COLOR),
             ('eve', 2, 'Eve-Neu', cfg.NEU_COLOR),
             ('mor', 1, 'Mor-Neg', '#922B21'),
             ('mor', 2, 'Mor-Neu', '#1A5276')]

subj_means = {}
for subj, sessions in trials_data.items():
    subj_means[subj] = {}
    for sk, cond, *_ in cond_keys:
        vals = [t['score'] for t in sessions.get(sk, [])
                if not t['rejected'] and t['label'] == cond
                and t['score'] is not None and not np.isnan(t['score'])]
        if vals:
            subj_means[subj][(sk, cond)] = np.mean(vals)

valid = [s for s in subj_means if all((sk,c) in subj_means[s] for sk,c,*_ in cond_keys)]

# IQR outlier removal on paired (Eve − Mor) difference
if len(valid) >= 4:
    eve_m = [np.nanmean([subj_means[s].get(('eve',1), np.nan), subj_means[s].get(('eve',2), np.nan)]) for s in valid]
    mor_m = [np.nanmean([subj_means[s].get(('mor',1), np.nan), subj_means[s].get(('mor',2), np.nan)]) for s in valid]
    _, _, valid, n_removed = _iqr_filter_paired(eve_m, mor_m, valid, IQR_OUTLIER_MULTIPLIER)
    if n_removed:
        print(f'IQR outlier removal: {n_removed} subject(s) removed from 4-condition boxplot')

if len(valid) < 2:
    print('Not enough subjects')
else:
    data_cols = [[subj_means[s][(sk,c)] for s in valid] for sk,c,*_ in cond_keys]
    n_trials  = [sum(1 for s in valid for t in trials_data[s].get(sk,[])
                     if not t['rejected'] and t['label']==c) for sk,c,*_ in cond_keys]
    fig, ax = plt.subplots(figsize=(10, 6))
    bp = ax.boxplot(data_cols, positions=[1,2,3,4], widths=0.45, patch_artist=True, showfliers=False)
    for patch,(sk,c,lbl,clr) in zip(bp['boxes'], cond_keys):
        patch.set_facecolor(clr); patch.set_alpha(0.65); patch.set_edgecolor('gray')
    for med in bp['medians']: med.set(color='black', linewidth=2)
    for s in valid:
        j = np.random.uniform(-0.07, 0.07)
        for (pn,pu),sk in [((1,2),'eve'),((3,4),'mor')]:
            vn = subj_means[s].get((sk,1)); vu = subj_means[s].get((sk,2))
            if vn is not None and vu is not None:
                lc = UP_COLOR if vu>=vn else DOWN_COLOR
                ax.plot([pn+j,pu+j],[vn,vu],color=lc,alpha=0.45,lw=1.2)
                ax.scatter(pn+j,vn,color=cond_keys[0 if pn==1 else 2][3],edgecolors='k',s=45,zorder=4,lw=0.5)
                ax.scatter(pu+j,vu,color=cond_keys[1 if pu==2 else 3][3],edgecolors='k',s=45,zorder=4,lw=0.5)
    for ca,cb,x1,x2 in [(data_cols[0],data_cols[1],1,2),(data_cols[2],data_cols[3],3,4)]:
        try:    _,p = stats.wilcoxon(ca,cb); tl='Wilcoxon'
        except: _,p = stats.ttest_rel(ca,cb); tl='Paired t'
        y_top = max(max(ca),max(cb))*1.08
        ax.set_ylim(top=max(ax.get_ylim()[1], y_top*1.25))
        _bracket(ax,x1,x2,y_top,p,tl)
    ax.set_xticks([1,2,3,4])
    ax.set_xticklabels([f'{lbl}\n(n={nt})' for (_,_,lbl,_),nt in zip(cond_keys,n_trials)])
    ax.set_ylabel(YLABEL, fontsize=11)
    ax.set_title(f'Airflow Score by Condition  (N={len(valid)} subjects)', fontweight='bold')
    ax.axvline(2.5, color='lightgray', linestyle='--', lw=1)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Eve vs Mor — Negative trials (objective label) ────────────────────────────
eve_scores, mor_scores, valid_subjs = [], [], []

for subj, sessions in sorted(trials_data.items()):
    s = {}
    for sk in ['eve', 'mor']:
        vals = [t['score'] for t in sessions.get(sk, [])
                if not t['rejected'] and t['label'] == 1
                and t['score'] is not None and not np.isnan(t['score'])]
        if vals: s[sk] = np.mean(vals)
    if 'eve' in s and 'mor' in s:
        eve_scores.append(s['eve']); mor_scores.append(s['mor']); valid_subjs.append(subj)

# IQR outlier removal on paired (Eve − Mor) difference
eve_scores, mor_scores, valid_subjs, n_removed = _iqr_filter_paired(
    eve_scores, mor_scores, valid_subjs, IQR_OUTLIER_MULTIPLIER)
if n_removed:
    print(f'IQR outlier removal: {n_removed} subject(s) removed (paired Eve−Mor difference)')

n = len(valid_subjs)
if n < 2:
    print('Not enough subjects')
else:
    try:    _, p = stats.wilcoxon(eve_scores, mor_scores); tl = 'Wilcoxon'
    except: _, p = stats.ttest_rel(eve_scores, mor_scores); tl = 'Paired t'
    fig, ax = plt.subplots(figsize=(6, 6))
    bp = ax.boxplot([eve_scores, mor_scores], positions=[1,2], widths=0.4,
                    patch_artist=True, showfliers=False)
    for patch, clr in zip(bp['boxes'], [EVE_COLOR, MOR_COLOR]):
        patch.set_facecolor(clr); patch.set_alpha(0.6); patch.set_edgecolor('gray')
    for med in bp['medians']: med.set(color='black', linewidth=2)
    for ev, mo, subj in zip(eve_scores, mor_scores, valid_subjs):
        j = np.random.uniform(-0.06, 0.06)
        lc = UP_COLOR if mo >= ev else DOWN_COLOR
        ax.plot([1+j,2+j],[ev,mo],color=lc,alpha=0.55,linewidth=1.2)
        ax.scatter(1+j,ev,color=EVE_COLOR,edgecolors='k',s=55,zorder=4,linewidths=0.5)
        ax.scatter(2+j,mo,color=MOR_COLOR,edgecolors='k',s=55,zorder=4,linewidths=0.5)
    h = (ax.get_ylim()[1]-ax.get_ylim()[0])*0.015
    y_top = max(max(eve_scores),max(mor_scores))*1.08
    ax.set_ylim(top=max(ax.get_ylim()[1], y_top*1.25))
    ax.plot([1,1,2,2],[y_top,y_top+h,y_top+h,y_top],lw=1.2,color='k')
    ax.text(1.5,y_top+h*1.5,f'{_sig(p)}  {tl}  p={p:.3f}',
            ha='center',va='bottom',fontsize=9,fontweight='bold')
    ax.set_xticks([1,2])
    ax.set_xticklabels([f'Evening\n(n={n})',f'Morning\n(n={n})'],fontsize=11)
    ax.set_ylabel(YLABEL, fontsize=11)
    ax.set_title(f'Airflow Score — Negative trials\nEvening vs Morning  (N={n} subjects)',
                 fontweight='bold')
    ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    plt.tight_layout(); plt.show()
    print(f'Eve mean={np.mean(eve_scores):.3f}  Mor mean={np.mean(mor_scores):.3f}  p={p:.3f}')
    print(f'Subjects (Eve, Mor): ' + ', '.join(f'{s}({e:.2f},{m:.2f})' for s,e,m in zip(valid_subjs,eve_scores,mor_scores)))